# CNN Architectures

## Objectives
- Understand VGG, ResNet, DenseNet, MobileNet
- Learn residual connections
- Build ResNet block
- Compare architectures
- Understand architectural evolution

## Introduction
This notebook covers the most important CNN architectures that shaped modern deep learning.

## What We're About to Do

The code below imports essential libraries. These libraries provide the foundational tools for tensor operations and neural network construction. Pay attention to what each import provides – understanding dependencies helps you know what's available for solving problems.


In [ ]:
# Import necessary libraries for tensor operations and deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision.models import resnet18, vgg16, mobilenet_v2

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


In [ ]:
# Visualize the results
plt.figure(figsize=(12, 4))
plt.plot(range(len(result)), result, marker='o', linestyle='-', linewidth=2)
plt.title('Results Visualization', fontsize=14, fontweight='bold')
plt.xlabel('Index')
plt.ylabel('Value')
plt.grid(True, alpha=0.3)
plt.show()

print('Visualization shows the pattern and distribution of results.')


In [ ]:
# Define a custom function with detailed implementation
## 1. VGG Architecture

class VGGBlock(nn.Module):
    def __init__(self, in_channels, out_channels, num_convs):
        super().__init__()
        layers = []
# Iterate through batches of data
        for i in range(num_convs):
            layers.append(nn.Conv2d(in_channels if i == 0 else out_channels, 
                                   out_channels, kernel_size=3, padding=1))
            layers.append(nn.ReLU(inplace=True))
        self.block = nn.Sequential(*layers)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
    
    def forward(self, x):
        x = self.block(x)
        x = self.pool(x)
        return x

class SimpleVGG(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            VGGBlock(3, 64, 2),
            VGGBlock(64, 128, 2),
            VGGBlock(128, 256, 2)
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(256, num_classes)
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = x.flatten(1)
        x = self.classifier(x)
        return x

vgg = SimpleVGG()
print("SimpleVGG created")
# Iterate through batches of data
print(f"Total params: {sum(p.numel() for p in vgg.parameters()):,}")


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
# Define a custom function with detailed implementation
## 2. Residual Connection

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                              stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                              padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Skip connection
        self.skip = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        identity = x
        
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        # Skip connection
        out = out + self.skip(identity)
        out = F.relu(out)
        
        return out

print("ResidualBlock defined")
print("\nKey idea: Add input to output (skip connection)")
print("Enables training of very deep networks")


In [ ]:
# Define a custom function with detailed implementation
## 3. Simple ResNet

class SimpleResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.layer1 = nn.Sequential(
            ResidualBlock(64, 64),
            ResidualBlock(64, 64)
        )
        
        self.layer2 = nn.Sequential(
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 128)
        )
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.avgpool(x)
        x = x.flatten(1)
        x = self.fc(x)
        return x

resnet = SimpleResNet()
print("SimpleResNet created")
# Iterate through batches of data
print(f"Total params: {sum(p.numel() for p in resnet.parameters()):,}")


In [ ]:
# Define a custom function with detailed implementation
## 4. Dense Connection (DenseNet idea)

class DenseBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    
    def forward(self, x):
        out = self.conv1(F.relu(self.bn1(x)))
        # Concatenate input with output
        return torch.cat([x, out], 1)

class SimpleDenseNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        # Dense block
        in_ch = 64
        self.dense = nn.Sequential()
# Iterate through batches of data
        for _ in range(3):
            self.dense.append(DenseBlock(in_ch, 32))
            in_ch += 32
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(in_ch, num_classes)
    
    def forward(self, x):
        x = self.pool(self.conv1(x))
        x = self.dense(x)
        x = self.avgpool(x)
        x = x.flatten(1)
        x = self.fc(x)
        return x

densenet = SimpleDenseNet()
print("SimpleDenseNet created")
# Iterate through batches of data
print(f"Total params: {sum(p.numel() for p in densenet.parameters()):,}")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Set up the neural network model architecture
## 5. Architecture Comparison

# Test forward pass
x = torch.randn(1, 3, 224, 224)

models = {
    'SimpleVGG': vgg,
    'SimpleResNet': resnet,
    'SimpleDenseNet': densenet
}

print("\nModel Comparison:")
print(f"{'Model':<20} {'Params':>12} {'Output Shape':<15}")
print("="*50)

# Iterate through batches of data
for name, model in models.items():
    model.eval()
    with torch.no_grad():
        out = model(x)
# Update model parameters based on computed gradients
# Iterate through batches of data
    params = sum(p.numel() for p in model.parameters())
    print(f"{name:<20} {params:>12,} {str(out.shape):<15}")


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
# Execute the training loop with proper tracking
## 6. Skip Connections Visualization

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# VGG (no skip)
axes[0].text(0.5, 0.5, 'VGG\nNo Skip Connections\nDeeper = Harder to train', 
            ha='center', va='center', fontsize=14, bbox=dict(boxstyle='round', facecolor='lightblue'))
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].axis('off')
axes[0].set_title('VGG Architecture')

# ResNet (skip)
axes[1].text(0.5, 0.5, 'ResNet\nWith Skip Connections\nLearn residuals, not identity', 
            ha='center', va='center', fontsize=14, bbox=dict(boxstyle='round', facecolor='lightgreen'))
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].axis('off')
axes[1].set_title('ResNet Architecture')

# DenseNet (dense)
axes[2].text(0.5, 0.5, 'DenseNet\nDense Connections\nConcatenate features', 
            ha='center', va='center', fontsize=14, bbox=dict(boxstyle='round', facecolor='lightyellow'))
axes[2].set_xlim(0, 1)
axes[2].set_ylim(0, 1)
axes[2].axis('off')
axes[2].set_title('DenseNet Architecture')

plt.tight_layout()
plt.show()


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
## 7. Using Pretrained Models

# Load pretrained
resnet18_pretrained = resnet18(pretrained=False)  # Set to True to download
print("ResNet18 model loaded")
print(f"Input: (batch, 3, 224, 224)")
print(f"Output: (batch, 1000)  [ImageNet classes]")

# Modify for 10 classes
resnet18_pretrained.fc = nn.Linear(512, 10)
print(f"\nModified for 10 classes")
print(f"New output: (batch, 10)")

In [ ]:
# Execute code with detailed step-by-step process
## 8. Architecture Evolution Timeline

timeline = {
    '2012': 'AlexNet - Deep CNN breakthrough',
    '2014': 'VGGNet - 16/19 layer networks',
    '2015': 'ResNet - Skip connections, 152 layers',
    '2016': 'DenseNet - Dense connections',
    '2017': 'MobileNet - Efficient mobile networks',
    '2018': 'EfficientNet - Scaling methodology',
    '2020': 'Vision Transformer - Attention-based'
}

print("\nCNN Architecture Evolution:")
print("="*50)
# Iterate through batches of data
for year, desc in timeline.items():
    print(f"{year}: {desc}")
print("="*50)


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
# Execute the training loop with proper tracking
## 9. Key Architectural Innovations

innovations = """
KEY INNOVATIONS:

1. SKIP CONNECTIONS (ResNet):
   - Enable training of very deep networks
   - Help with vanishing gradient problem
   - Learn residuals instead of absolute functions

2. BATCH NORMALIZATION:
   - Normalize activations between layers
   - Allows higher learning rates
   - Reduces internal covariate shift

3. BOTTLENECK BLOCKS:
   - 1x1 conv to reduce dimensions
# Iterate through batches of data
   - 3x3 conv for computation
   - 1x1 conv to expand dimensions
   - Efficient parameter usage

4. DENSE CONNECTIONS:
   - Connect every layer to every other layer
   - Encourage feature reuse
   - Better gradient flow
   - More efficient than ResNet

5. DEPTHWISE SEPARABLE CONVOLUTION:
   - Separate spatial and channel operations
   - Reduce parameters and computation
# Iterate through batches of data
   - Essential for mobile networks
"""

print(innovations)


## What We're About to Do

The code below imports essential libraries. These libraries provide the foundational tools for tensor operations and neural network construction. Pay attention to what each import provides – understanding dependencies helps you know what's available for solving problems.


In [ ]:
# Import necessary libraries for tensor operations and deep learning
## 10. Summary Table

import pandas as pd

data = {
    'Architecture': ['AlexNet', 'VGG', 'ResNet', 'DenseNet', 'MobileNet'],
    'Year': [2012, 2014, 2015, 2016, 2017],
    'Depth': ['8 layers', '16-19 layers', '50-152 layers', '121 layers', '28-53 layers'],
    'Key Innovation': ['ReLU, Dropout', 'Simplicity', 'Skip connections', 'Dense connections', 'Efficiency'],
    'Best For': ['General vision', 'Baseline', 'High accuracy', 'Accuracy + efficiency', 'Mobile devices']
}

df = pd.DataFrame(data)
print("\nArchitecture Comparison:")
print(df.to_string(index=False))


## What We're About to Do

The code below imports essential libraries. These libraries provide the foundational tools for tensor operations and neural network construction. Pay attention to what each import provides – understanding dependencies helps you know what's available for solving problems.


## 🎯 Key Takeaways

✅ **Understanding fundamentals is crucial** – The concepts covered here form the foundation for all advanced deep learning techniques.

✅ **Each component has a specific purpose** – Whether it's data loading, model architecture, or optimization, every piece serves a distinct function in the pipeline.

✅ **Experimentation drives learning** – Don't just read the code; modify it, break it, and see what happens. That's how intuition develops.

✅ **Deep learning is iterative** – Success comes from systematically trying approaches, measuring results, and refining based on evidence.

✅ **Connect concepts, don't memorize** – Understanding how PyTorch tensors relate to autograd, which relates to neural networks, which connects to training loops, is far more valuable than memorizing individual APIs.

✅ **Performance matters in practice** – Once you understand the theory, optimizing for speed, memory, and scalability becomes crucial for real-world applications.


In [ ]:
# Import necessary libraries for tensor operations and deep learning
## Key Takeaways
- Skip connections enable training of very deep networks
- ResNet introduced the residual learning framework
- DenseNet improves efficiency through dense connections
# Iterate through batches of data
- Batch normalization is essential for modern architectures
- Architecture choice impacts both accuracy and efficiency

## Interview Q&A

**Q1: Why are skip connections important?**
Skip connections allow gradients to flow directly to earlier layers, solving the vanishing gradient problem. They also encourage the network to learn residuals (difference from identity) rather than absolute functions.

**Q2: What's the advantage of DenseNet over ResNet?**
DenseNet concatenates features from all previous layers, promoting feature reuse and improving gradient flow. It achieves similar accuracy with fewer parameters and more computational efficiency.

**Q3: Why does ResNet work so well?**
The residual learning framework is particularly effective because it's easier to optimize - networks learn the residual (difference) rather than the mapping itself. This makes even very deep networks trainable.

## References
- [ResNet Paper](https://arxiv.org/abs/1512.03385)
- [DenseNet Paper](https://arxiv.org/abs/1608.06993)
- [MobileNet Paper](https://arxiv.org/abs/1704.04861)
